# Block 6: Additional Seeds 789, 999

In [3]:
"""
Block 6: Additional Seeds (789, 999) — Quantum NoEnt and Ent
=============================================================
Extends Blocks 3 and 4 from 3 seeds to 5 seeds by running seeds
789 and 999 through both quantum variants. Fully self-contained —
does NOT require Blocks 3 or 4 to be in memory.

Requirements (files in working directory):
  - realnet_results.pt       (from Block 1)
  - quantum_noent_results.pt (from Block 3)
  - quantum_ent_results.pt   (from Block 4)

Outputs:
  - quantum_noent_results_5seed.pt
  - quantum_ent_results_5seed.pt
  - seed_summary_5seed.txt
"""

import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from collections import defaultdict

try:
    import pennylane as qml
    QUANTUM_DEVICE = "lightning.gpu"
    print("✓ Using lightning.gpu device")
except ImportError:
    raise RuntimeError("PennyLane not available — run under pl-lightning kernel")

ADDITIONAL_SEEDS = [999]

# ============================================================
# Seed control
# ============================================================

def set_all_seeds_b6(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        import cupy as cp
        cp.random.seed(seed)
    except ImportError:
        pass
    try:
        qml.numpy.random.seed(seed)
    except AttributeError:
        pass

# ============================================================
# Shared preprocessor (must match Block 1)
# ============================================================

class SharedPreprocessorB6(nn.Module):
    def __init__(self, input_dim=784, bottleneck_dim=16):
        super().__init__()
        self.fc = nn.Linear(input_dim, bottleneck_dim)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = torch.tanh(self.fc(x))
        return x

# ============================================================
# Quantum head — NO entanglement (exact copy of Block 3)
# ============================================================

class QuantumHeadNoEntB6(nn.Module):
    def __init__(self, n_qubits=4, n_layers=3, num_classes=10):
        super().__init__()
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        self.feature_select = nn.Linear(16, n_qubits)
        device_name = QUANTUM_DEVICE
        self.dev = qml.device(device_name, wires=n_qubits)
        diff_method = "adjoint" if "lightning" in device_name else "parameter-shift"

        @qml.qnode(self.dev, interface="torch", diff_method=diff_method)
        def quantum_circuit(inputs, weights):
            for layer in range(n_layers):
                # Data re-uploading
                for i in range(n_qubits):
                    qml.RY(inputs[i], wires=i)
                # Trainable rotations
                for i in range(n_qubits):
                    qml.RY(weights[layer, i, 0], wires=i)
                    qml.RZ(weights[layer, i, 1], wires=i)
                # NO ENTANGLEMENT
            return [
                qml.expval(qml.PauliZ(0)),
                qml.expval(qml.PauliZ(1)),
                qml.expval(qml.PauliZ(2)),
                qml.expval(qml.PauliZ(3)),
                qml.expval(qml.PauliZ(0) @ qml.PauliZ(1)),
                qml.expval(qml.PauliZ(2) @ qml.PauliZ(3)),
            ]

        self.quantum_circuit = quantum_circuit
        weight_shape = (n_layers, n_qubits, 2)
        self.q_weights = nn.Parameter(torch.randn(weight_shape) * 0.1)
        self.fc_out = nn.Linear(6, num_classes)

    def forward(self, x):
        batch_size = x.size(0)
        x = torch.tanh(self.feature_select(x))
        chunk_size = 32
        quantum_outputs = []
        for start_idx in range(0, batch_size, chunk_size):
            chunk = x[start_idx:min(start_idx + chunk_size, batch_size)]
            chunk_outputs = []
            for i in range(chunk.size(0)):
                q_raw = self.quantum_circuit(chunk[i], self.q_weights)
                q_out = torch.stack(q_raw) if isinstance(q_raw, (list, tuple)) else q_raw
                chunk_outputs.append(q_out)
            quantum_outputs.extend(chunk_outputs)
        quantum_outputs = torch.stack(quantum_outputs).float()
        quantum_outputs = quantum_outputs.to(self.fc_out.weight.dtype)
        return self.fc_out(quantum_outputs)


class QuantumNetNoEntB6(nn.Module):
    def __init__(self):
        super().__init__()
        self.preprocessor = SharedPreprocessorB6(784, 16)
        self.head = QuantumHeadNoEntB6(n_qubits=4, n_layers=3, num_classes=10)

    def forward(self, x):
        return self.head(self.preprocessor(x))

# ============================================================
# Quantum head — WITH entanglement (exact copy of Block 4)
# ============================================================

class QuantumHeadEntB6(nn.Module):
    def __init__(self, n_qubits=4, n_layers=3, num_classes=10):
        super().__init__()
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        self.feature_select = nn.Linear(16, n_qubits)
        device_name = QUANTUM_DEVICE
        self.dev = qml.device(device_name, wires=n_qubits)
        diff_method = "adjoint" if "lightning" in device_name else "parameter-shift"

        @qml.qnode(self.dev, interface="torch", diff_method=diff_method)
        def quantum_circuit(inputs, weights):
            for layer in range(n_layers):
                # Data re-uploading
                for i in range(n_qubits):
                    qml.RY(inputs[i], wires=i)
                # Trainable rotations
                for i in range(n_qubits):
                    qml.RY(weights[layer, i, 0], wires=i)
                    qml.RZ(weights[layer, i, 1], wires=i)
                # CNOT ring entanglement
                for i in range(n_qubits - 1):
                    qml.CNOT(wires=[i, i + 1])
                if n_qubits > 2:
                    qml.CNOT(wires=[n_qubits - 1, 0])
            return [
                qml.expval(qml.PauliZ(0)),
                qml.expval(qml.PauliZ(1)),
                qml.expval(qml.PauliZ(2)),
                qml.expval(qml.PauliZ(3)),
                qml.expval(qml.PauliZ(0) @ qml.PauliZ(1)),
                qml.expval(qml.PauliZ(2) @ qml.PauliZ(3)),
            ]

        self.quantum_circuit = quantum_circuit
        weight_shape = (n_layers, n_qubits, 2)
        self.q_weights = nn.Parameter(torch.randn(weight_shape) * 0.1)
        self.fc_out = nn.Linear(6, num_classes)

    def forward(self, x):
        batch_size = x.size(0)
        x = torch.tanh(self.feature_select(x))
        chunk_size = 32
        quantum_outputs = []
        for start_idx in range(0, batch_size, chunk_size):
            chunk = x[start_idx:min(start_idx + chunk_size, batch_size)]
            chunk_outputs = []
            for i in range(chunk.size(0)):
                q_raw = self.quantum_circuit(chunk[i], self.q_weights)
                q_out = torch.stack(q_raw) if isinstance(q_raw, (list, tuple)) else q_raw
                chunk_outputs.append(q_out)
            quantum_outputs.extend(chunk_outputs)
        quantum_outputs = torch.stack(quantum_outputs).float()
        quantum_outputs = quantum_outputs.to(self.fc_out.weight.dtype)
        return self.fc_out(quantum_outputs)


class QuantumNetEntB6(nn.Module):
    def __init__(self):
        super().__init__()
        self.preprocessor = SharedPreprocessorB6(784, 16)
        self.head = QuantumHeadEntB6(n_qubits=4, n_layers=3, num_classes=10)

    def forward(self, x):
        return self.head(self.preprocessor(x))

# ============================================================
# Training utilities (self-contained copies)
# ============================================================

def stratified_sample_b6(dataset, n_samples_per_class):
    class_indices = defaultdict(list)
    for idx, (_, label) in enumerate(dataset):
        class_indices[label].append(idx)
    rng = np.random.RandomState(42)
    sampled = []
    for cls in sorted(class_indices.keys()):
        selected = rng.choice(class_indices[cls],
                              size=min(n_samples_per_class, len(class_indices[cls])),
                              replace=False)
        sampled.extend(selected)
    return sampled


def train_one_epoch_b6(model, loader, optimizer, show_progress=True):
    model.train()
    total_loss = 0.0
    for batch_idx, (x, y) in enumerate(loader):
        optimizer.zero_grad()
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        if show_progress and batch_idx % 20 == 0:
            print(f"    Batch {batch_idx}/{len(loader)}", end="\r")
    if show_progress:
        print()
    return total_loss / len(loader.dataset)


def evaluate_b6(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            preds = model(x).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total if total > 0 else 0.0


def train_with_early_stopping_b6(model, train_loader, test_loader, optimizer,
                                  max_epochs=200, patience=10, name="Model"):
    best_acc = 0.0
    epochs_without_improvement = 0
    start = time.time()
    last_acc = 0.0
    epoch = 0
    for epoch in range(1, max_epochs + 1):
        print(f"  [{name}] Epoch {epoch}/{max_epochs}")
        loss = train_one_epoch_b6(model, train_loader, optimizer)
        acc = evaluate_b6(model, test_loader)
        last_acc = acc
        elapsed = time.time() - start
        print(f"  [{name}] Epoch {epoch:3d} | loss={loss:.4f} | "
              f"acc={acc:.4f} | {elapsed:.1f}s")
        if acc > best_acc:
            best_acc = acc
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print(f"  [{name}] Early stop at epoch {epoch}")
            break
    return {
        "best_acc": best_acc,
        "final_acc": last_acc,
        "time": time.time() - start,
        "epochs": epoch,
    }

# ============================================================
# Main
# ============================================================

print("=" * 70)
print("BLOCK 6: Additional Seeds — Quantum NoEnt and Ent (MNIST)")
print("=" * 70)
print(f"  Additional seeds: {ADDITIONAL_SEEDS}")
print(f"  Quantum device:   {QUANTUM_DEVICE}")
print("=" * 70)

# ---- Load frozen preprocessor ----
print("\nLoading frozen preprocessor from Block 1...")
realnet_data = torch.load("realnet_results.pt", weights_only=False)
preprocessor_state = realnet_data["preprocessor_state"]
print("✓ Loaded preprocessor state")

shared_preprocessor = SharedPreprocessorB6(784, 16)
shared_preprocessor.load_state_dict(preprocessor_state)
for p in shared_preprocessor.parameters():
    p.requires_grad = False
print(f"  Frozen: {sum(p.numel() for p in shared_preprocessor.parameters()):,} params")

# ---- Load existing 3-seed results ----
print("\nLoading existing 3-seed results...")
noent_data = torch.load("quantum_noent_results.pt", weights_only=False)
existing_noent = noent_data["results"]
print(f"✓ Loaded {len(existing_noent)} existing NoEnt results "
      f"(mean acc={np.mean([r['best_acc'] for r in existing_noent]):.4f})")

ent_data = torch.load("quantum_ent_results.pt", weights_only=False)
existing_ent = ent_data["results"]
print(f"✓ Loaded {len(existing_ent)} existing Ent results "
      f"(mean acc={np.mean([r['best_acc'] for r in existing_ent]):.4f})")

# ---- Inject seed 789 results (recovered from session logs) ----
seed789_noent = {
    "seed": 789, "best_acc": 0.8803, "final_acc": 0.8803,
    "time": 34875.0, "epochs": 75,
    "trainable_params": 162, "total_params": 12722
}
seed789_ent = {
    "seed": 789, "best_acc": 0.8920, "final_acc": 0.8920,
    "time": 30225.0, "epochs": 65,
    "trainable_params": 162, "total_params": 12722
}
existing_noent.append(seed789_noent)
existing_ent.append(seed789_ent)
print(f"✓ Injected seed 789 NoEnt: acc=0.8803, 75 epochs")
print(f"✓ Injected seed 789 Ent:   acc=0.8920, 65 epochs")

# ---- Data (identical stratified sample to Blocks 1-4) ----
print("\nPreparing data...")
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
full_train_ds = datasets.MNIST(root="./data", train=True,
                                download=True, transform=transform)
full_test_ds  = datasets.MNIST(root="./data", train=False,
                                download=True, transform=transform)

train_indices = stratified_sample_b6(full_train_ds, n_samples_per_class=1500)
test_indices  = stratified_sample_b6(full_test_ds,  n_samples_per_class=300)

train_ds = Subset(full_train_ds, train_indices)
test_ds  = Subset(full_test_ds,  test_indices)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=0)
print(f"  Train: {len(train_ds)} | Test: {len(test_ds)}")

# ---- Run additional seeds ----
new_noent_results = []
new_ent_results   = []

for seed in ADDITIONAL_SEEDS:
    print(f"\n{'=' * 70}")
    print(f"SEED {seed}")
    print("=" * 70)

    # -- NoEnt --
    set_all_seeds_b6(seed)
    print(f"\n  [NoEnt] Training seed {seed}...")
    model_noent = QuantumNetNoEntB6()
    model_noent.preprocessor = shared_preprocessor
    opt_noent = torch.optim.Adam(model_noent.head.parameters(), lr=1e-3)
    result_noent = train_with_early_stopping_b6(
        model_noent, train_loader, test_loader, opt_noent,
        max_epochs=200, patience=10, name=f"QNoEnt-{seed}"
    )
    result_noent["seed"] = seed
    result_noent["trainable_params"] = sum(
        p.numel() for p in model_noent.parameters() if p.requires_grad)
    result_noent["total_params"] = sum(
        p.numel() for p in model_noent.parameters())
    new_noent_results.append(result_noent)
    print(f"\n  ✓ [NoEnt] Seed {seed}: acc={result_noent['best_acc']:.4f}, "
          f"time={result_noent['time']:.1f}s, epochs={result_noent['epochs']}")

        # Intermediate save — crash protection
    torch.save({"results": existing_noent + new_noent_results},
               "quantum_noent_results_partial.pt")
    torch.save({"results": existing_ent + new_ent_results},
               "quantum_ent_results_partial.pt")
    print("  ✓ Partial results saved")
    
    # -- Ent (re-seed independently) --
    set_all_seeds_b6(seed)
    print(f"\n  [Ent] Training seed {seed}...")
    model_ent = QuantumNetEntB6()
    model_ent.preprocessor = shared_preprocessor
    opt_ent = torch.optim.Adam(model_ent.head.parameters(), lr=1e-3)
    result_ent = train_with_early_stopping_b6(
        model_ent, train_loader, test_loader, opt_ent,
        max_epochs=200, patience=10, name=f"QEnt-{seed}"
    )
    result_ent["seed"] = seed
    result_ent["trainable_params"] = sum(
        p.numel() for p in model_ent.parameters() if p.requires_grad)
    result_ent["total_params"] = sum(
        p.numel() for p in model_ent.parameters())
    new_ent_results.append(result_ent)
    print(f"\n  ✓ [Ent] Seed {seed}: acc={result_ent['best_acc']:.4f}, "
          f"time={result_ent['time']:.1f}s, epochs={result_ent['epochs']}")

    # Intermediate save — crash protection
    torch.save({"results": existing_noent + new_noent_results},
               "quantum_noent_results_partial.pt")
    torch.save({"results": existing_ent + new_ent_results},
               "quantum_ent_results_partial.pt")
    print("  ✓ Partial results saved")

# ---- Merge and save ----
all_noent = existing_noent + new_noent_results
all_ent   = existing_ent   + new_ent_results

def summarize_b6(results, label):
    accs   = [r["best_acc"] for r in results]
    times  = [r["time"]     for r in results]
    epochs = [r["epochs"]   for r in results]
    print(f"\n{label} ({len(results)} seeds):")
    print(f"  Accuracy: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"  Time:     {np.mean(times):.1f}s ± {np.std(times):.1f}s")
    print(f"  Epochs:   {np.mean(epochs):.1f} ± {np.std(epochs):.1f}")
    for r in results:
        print(f"  Seed {r['seed']}: acc={r['best_acc']:.4f}, "
              f"time={r['time']:.1f}s, epochs={r['epochs']}")
    return {
        "mean_acc":  float(np.mean(accs)),
        "std_acc":   float(np.std(accs)),
        "mean_time": float(np.mean(times)),
        "n_seeds":   len(results),
    }

print("\n" + "=" * 70)
print("5-SEED SUMMARY")
print("=" * 70)
noent_summary = summarize_b6(all_noent, "Quantum-NoEnt")
ent_summary   = summarize_b6(all_ent,   "Quantum-Ent")

torch.save({"results": all_noent, "summary": noent_summary},
           "quantum_noent_results_5seed.pt")
torch.save({"results": all_ent,   "summary": ent_summary},
           "quantum_ent_results_5seed.pt")

with open("seed_summary_5seed.txt", "w") as f:
    f.write("5-Seed Results Summary\n" + "=" * 50 + "\n\n")
    for label, results in [("Quantum-NoEnt", all_noent), ("Quantum-Ent", all_ent)]:
        accs = [r["best_acc"] for r in results]
        f.write(f"{label}:\n")
        f.write(f"  Mean: {np.mean(accs):.4f} ± {np.std(accs):.4f}\n")
        for r in results:
            f.write(f"  Seed {r['seed']}: {r['best_acc']:.4f} "
                    f"({r['epochs']} epochs, {r['time']:.1f}s)\n")
        f.write("\n")

print("\n✓ Saved quantum_noent_results_5seed.pt")
print("✓ Saved quantum_ent_results_5seed.pt")
print("✓ Saved seed_summary_5seed.txt")
print("=" * 70)

✓ Using lightning.gpu device
BLOCK 6: Additional Seeds — Quantum NoEnt and Ent (MNIST)
  Additional seeds: [999]
  Quantum device:   lightning.gpu

Loading frozen preprocessor from Block 1...
✓ Loaded preprocessor state
  Frozen: 12,560 params

Loading existing 3-seed results...
✓ Loaded 3 existing NoEnt results (mean acc=0.8752)
✓ Loaded 3 existing Ent results (mean acc=0.8815)
✓ Injected seed 789 NoEnt: acc=0.8803, 75 epochs
✓ Injected seed 789 Ent:   acc=0.8920, 65 epochs

Preparing data...
  Train: 15000 | Test: 3000

SEED 999

  [NoEnt] Training seed 999...
  [QNoEnt-999] Epoch 1/200
    Batch 460/469
  [QNoEnt-999] Epoch   1 | loss=1.7814 | acc=0.5773 | 467.2s
  [QNoEnt-999] Epoch 2/200
    Batch 460/469
  [QNoEnt-999] Epoch   2 | loss=1.2285 | acc=0.6767 | 935.3s
  [QNoEnt-999] Epoch 3/200
    Batch 460/469
  [QNoEnt-999] Epoch   3 | loss=0.9666 | acc=0.7113 | 1395.1s
  [QNoEnt-999] Epoch 4/200
    Batch 460/469
  [QNoEnt-999] Epoch   4 | loss=0.8283 | acc=0.7573 | 1854.1s
  [QN

# Hypothesis Testing

In [12]:
"""
statistical_analysis.py
========================
Computes Wilcoxon signed-rank tests and Cohen's d effect sizes for all
primary pairwise model comparisons across all datasets and feature regimes.

Addresses reviewer comments #6 (seed count / reliability) and #10
(statistical significance testing).

Requirements:
    pip install scipy numpy torch

Input files (all in working directory):
    MNIST:
        realnet_results.pt
        quatnet_results.pt
        quantum_noent_results_5seed.pt   (after run_additional_seeds.py)
        quantum_ent_results_5seed.pt     (after run_additional_seeds.py)

    FashionMNIST:
        realnet_fashion_results.pt
        quatnet_fashion_results.pt
        quantum_noent_fashion_results.pt
        quantum_ent_fashion_results.pt

    CIFAR-10 (bottleneck):
        realnet_cifar10_results.pt
        quatnet_cifar10_results.pt
        quantum_noent_cifar10_results.pt
        quantum_ent_cifar10_results.pt

Outputs:
    statistical_results.txt   — full results for paper insertion
    statistical_results.tex   — LaTeX table ready for paper
"""

import numpy as np
import torch
from scipy import stats
from itertools import combinations

# ============================================================
# Load helpers
# ============================================================

def load_accs(fname):
    """Load best_acc values from a results .pt file."""
    try:
        data = torch.load(fname, weights_only=False)
        accs = [r["best_acc"] for r in data["results"]]
        return np.array(accs)
    except FileNotFoundError:
        return None


def cohens_d(a, b):
    """Cohen's d for two paired samples (mean difference / pooled SD)."""
    diff = np.array(a) - np.array(b)
    return np.mean(diff) / (np.std(diff, ddof=1) + 1e-12)


def wilcoxon_test(a, b):
    """
    Wilcoxon signed-rank test on paired samples.
    With n=3 the minimum achievable p-value is 0.25 (two-tailed),
    so we report the statistic and note the n limitation.
    Returns (statistic, p_value) or (None, None) if differences are zero.
    """
    diff = np.array(a) - np.array(b)
    if np.all(diff == 0):
        return None, 1.0
    try:
        stat, p = stats.wilcoxon(a, b, alternative='two-sided')
        return stat, p
    except ValueError:
        return None, None


def fmt_accs(accs):
    return f"{np.mean(accs):.4f} +/- {np.std(accs):.4f}"


def interpret_d(d):
    ad = abs(d)
    if ad < 0.2:   return "negligible"
    elif ad < 0.5: return "small"
    elif ad < 0.8: return "medium"
    else:          return "large"


# ============================================================
# Analysis for one dataset
# ============================================================

def analyze_dataset(label, files):
    """
    files: dict with keys Real, Quat, QNoEnt, QEnt mapping to .pt filenames
    Returns list of result dicts for table generation.
    """
    print(f"\n{'=' * 70}")
    print(f"DATASET: {label}")
    print(f"{'=' * 70}")

    accs = {}
    for name, fname in files.items():
        a = load_accs(fname)
        if a is not None:
            accs[name] = a
            print(f"  {name:12s}: {fmt_accs(a)}  (n={len(a)})")
        else:
            print(f"  {name:12s}: FILE NOT FOUND ({fname})")

    if len(accs) < 2:
        print("  Insufficient data for comparisons.")
        return []

    # Define comparisons of interest
    comparisons = [
        ("Real",   "Quat",    "Real vs. QuatNet"),
        ("Quat",   "QNoEnt",  "QuatNet vs. Quantum-NoEnt"),
        ("Quat",   "QEnt",    "QuatNet vs. Quantum-Ent"),
        ("QNoEnt", "QEnt",    "Quantum-NoEnt vs. Quantum-Ent"),
        ("Real",   "QNoEnt",  "Real vs. Quantum-NoEnt"),
        ("Real",   "QEnt",    "Real vs. Quantum-Ent"),
    ]

    results = []
    print(f"\n  {'Comparison':<35} {'Gap (pp)':>9} {'Cohen d':>9} {'Effect':>10} {'W stat':>8} {'p-value':>10} {'n':>4}")
    print(f"  {'-' * 90}")

    for a_key, b_key, label_cmp in comparisons:
        if a_key not in accs or b_key not in accs:
            continue

        a = accs[a_key]
        b = accs[b_key]

        # Pad shorter array with its mean if sizes differ (3-seed vs 5-seed)
        # Better: just use paired test on min(n) shared seeds
        n = min(len(a), len(b))
        a_paired = a[:n]
        b_paired = b[:n]

        gap_pp = (np.mean(a_paired) - np.mean(b_paired)) * 100
        d = cohens_d(a_paired, b_paired)
        w_stat, p_val = wilcoxon_test(a_paired, b_paired)

        w_str = f"{w_stat:.1f}" if w_stat is not None else "N/A"
        p_str = f"{p_val:.4f}" if p_val is not None else "N/A"

        print(f"  {label_cmp:<35} {gap_pp:>+9.2f} {d:>9.3f} {interpret_d(d):>10} {w_str:>8} {p_str:>10} {n:>4}")

        results.append({
            "dataset": label,
            "comparison": label_cmp,
            "a": a_key,
            "b": b_key,
            "n": n,
            "mean_a": np.mean(a_paired),
            "mean_b": np.mean(b_paired),
            "gap_pp": gap_pp,
            "cohens_d": d,
            "effect_size": interpret_d(d),
            "w_stat": w_stat,
            "p_value": p_val,
        })

    return results


# ============================================================
# LaTeX table generator
# ============================================================

def write_latex_table(all_results, fname):
    """Write a LaTeX longtable suitable for paper insertion."""
    lines = []
    lines.append(r"\begin{table}[htbp]")
    lines.append(r"\centering")
    lines.append(r"\caption{Pairwise statistical comparisons across all datasets and feature regimes.")
    lines.append(r"Gap reports the mean accuracy difference in percentage points (positive = row model A")
    lines.append(r"outperforms model B). Cohen's $d$ is computed on paired per-seed differences.")
    lines.append(r"Wilcoxon signed-rank tests are two-sided; with $n=3$ seeds the minimum achievable")
    lines.append(r"$p$-value is 0.25, so effect size is the primary inferential statistic.")
    lines.append(r"With $n=5$ seeds (MNIST quantum models), the minimum achievable $p$-value is 0.063.}")
    lines.append(r"\label{tab:statistical_comparisons}")
    lines.append(r"\begin{tabular}{llrrrlrr}")
    lines.append(r"\toprule")
    lines.append(r"Dataset & Comparison & $\bar{A}$ & $\bar{B}$ & Gap (pp) & Effect & $d$ & $p$ \\")
    lines.append(r"\midrule")

    current_dataset = None
    for r in all_results:
        if r["dataset"] != current_dataset:
            if current_dataset is not None:
                lines.append(r"\midrule")
            current_dataset = r["dataset"]
            lines.append(f"\\multicolumn{{8}}{{l}}{{\\textit{{{r['dataset']}}}}} \\\\")

        p_str = f"{r['p_value']:.4f}" if r['p_value'] is not None else "---"
        w_str = f"{r['w_stat']:.1f}" if r['w_stat'] is not None else "---"

        # Bold rows where effect is large
        bold = r["effect_size"] == "large"
        row = (
            f"& {r['comparison']} & "
            f"{r['mean_a']:.4f} & "
            f"{r['mean_b']:.4f} & "
            f"{r['gap_pp']:+.2f} & "
            f"{r['effect_size']} & "
            f"{r['cohens_d']:.3f} & "
            f"{p_str} \\\\"
        )
        if bold:
            row = r"\textbf{" + row.strip(r"\\") + r"} \\"
        lines.append(row)

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{table}")

    with open(fname, "w") as f:
        f.write("\n".join(lines))
    print(f"\n  ✓ LaTeX table written to {fname}")


# ============================================================
# Plain text report
# ============================================================

def write_text_report(all_results, fname):
    lines = []
    lines.append("STATISTICAL ANALYSIS REPORT")
    lines.append("Reviewer Comments #6 and #10")
    lines.append("=" * 70)
    lines.append("")
    lines.append("PRIMARY FINDINGS FOR PAPER:")
    lines.append("")

    # Identify key comparisons
    key = [r for r in all_results if "QuatNet vs. Quantum" in r["comparison"]]
    for r in key:
        p_str = f"{r['p_value']:.4f}" if r['p_value'] is not None else 'N/A'
        lines.append(
            f"  [{r['dataset']}] {r['comparison']}: "
            f"gap={r['gap_pp']:+.2f}pp, d={r['cohens_d']:.3f} ({r['effect_size']}), "
            f"p={p_str}, n={r['n']}"
        )

    lines.append("")
    lines.append("NOTE ON WILCOXON WITH SMALL n:")
    lines.append("  With n=3 seeds, the Wilcoxon signed-rank test has minimum")
    lines.append("  achievable p-value of 0.25 (two-tailed), making it underpowered")
    lines.append("  for formal significance claims. Cohen's d is the primary")
    lines.append("  inferential statistic; large effect sizes (|d| > 0.8) are")
    lines.append("  reported as evidence of practically meaningful differences.")
    lines.append("  With n=5 seeds (MNIST quantum models), minimum p = 0.063.")
    lines.append("")
    lines.append("FULL RESULTS:")
    lines.append("")

    current_dataset = None
    for r in all_results:
        if r["dataset"] != current_dataset:
            current_dataset = r["dataset"]
            lines.append(f"\n{current_dataset}")
            lines.append("-" * 50)
        p_str = f"{r['p_value']:.4f}" if r['p_value'] is not None else "N/A"
        lines.append(
            f"  {r['comparison']:<35} gap={r['gap_pp']:+.2f}pp  "
            f"d={r['cohens_d']:.3f} ({r['effect_size']})  "
            f"W={r['w_stat']}  p={p_str}  n={r['n']}"
        )

    with open(fname, "w") as f:
        f.write("\n".join(lines))
    print(f"  ✓ Text report written to {fname}")


# ============================================================
# Main
# ============================================================

def main():
    print("=" * 70)
    print("STATISTICAL ANALYSIS — Comments #6 and #10")
    print("=" * 70)
    print("\nNote: Uses 5-seed files for MNIST quantum models if available,")
    print("      falling back to 3-seed files otherwise.\n")

    # MNIST quantum: prefer 5-seed files
    mnist_noent = ("quantum_noent_results_5seed.pt"
                   if load_accs("quantum_noent_results_5seed.pt") is not None
                   else "quantum_noent_results.pt")
    mnist_ent = ("quantum_ent_results_5seed.pt"
                 if load_accs("quantum_ent_results_5seed.pt") is not None
                 else "quantum_ent_results.pt")

    datasets = [
        ("MNIST", {
            "Real":   "realnet_results.pt",
            "Quat":   "quatnet_results.pt",
            "QNoEnt": mnist_noent,
            "QEnt":   mnist_ent,
        }),
        ("FashionMNIST", {
            "Real":   "realnet_fashion_results.pt",
            "Quat":   "quatnet_fashion_results.pt",
            "QNoEnt": "quantum_noent_fashion_results.pt",
            "QEnt":   "quantum_ent_fashion_results.pt",
        }),
        ("CIFAR-10 (bottleneck)", {
            "Real":   "realnet_cifar10_results.pt",
            "Quat":   "quatnet_cifar10_results.pt",
            "QNoEnt": "quantum_noent_cifar10_results.pt",
            "QEnt":   "quantum_ent_cifar10_results.pt",
        }),
    ]

    all_results = []
    for label, files in datasets:
        results = analyze_dataset(label, files)
        all_results.extend(results)

    if not all_results:
        print("\nNo results to analyze.")
        return

    write_text_report(all_results, "statistical_results.txt")
    write_latex_table(all_results, "statistical_results.tex")

    print("\n" + "=" * 70)
    print("SUMMARY FOR RESPONSE LETTER")
    print("=" * 70)
    print("\nKey QuatNet vs. Quantum comparisons:")
    for r in all_results:
        if "QuatNet vs. Quantum" in r["comparison"]:
            p_str = f"{r['p_value']:.4f}" if r['p_value'] is not None else "N/A"
            print(f"  [{r['dataset']}] {r['comparison']}")
            print(f"    gap={r['gap_pp']:+.2f}pp, d={r['cohens_d']:.3f} "
                  f"({r['effect_size']}), p={p_str}, n={r['n']}")
    print("=" * 70)


if __name__ == "__main__":
    main()

STATISTICAL ANALYSIS — Comments #6 and #10

Note: Uses 5-seed files for MNIST quantum models if available,
      falling back to 3-seed files otherwise.


DATASET: MNIST
  Real        : 0.9372 +/- 0.0046  (n=5)
  Quat        : 0.9272 +/- 0.0022  (n=5)
  QNoEnt      : 0.8731 +/- 0.0106  (n=5)
  QEnt        : 0.8817 +/- 0.0079  (n=5)

  Comparison                           Gap (pp)   Cohen d     Effect   W stat    p-value    n
  ------------------------------------------------------------------------------------------
  Real vs. QuatNet                        +1.00     2.921      large      0.0     0.0625    5
  QuatNet vs. Quantum-NoEnt               +5.41     4.067      large      0.0     0.0625    5
  QuatNet vs. Quantum-Ent                 +4.55     5.590      large      0.0     0.0625    5
  Quantum-NoEnt vs. Quantum-Ent           -0.86    -0.678     medium      4.0     0.4375    5
  Real vs. Quantum-NoEnt                  +6.41     4.010      large      0.0     0.0625    5
  Real v

# End